In [ ]:
import math
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from pywaffle import Waffle

# ===== Knobs =====
MM = 1 / 25.4          # millimetre → inch conversion factor

FIG_WIDTH_MM  = 180    # total figure width  (mm)
FIG_HEIGHT_MM = 90     # total figure height (mm)
FONTSIZE        = 6    # body text, pt
HEADER_FONTSIZE = 7    # cohort headers + corner label, pt
WAFFLE_ROWS     = 5    # vertical cells per waffle

# Outer margins (fraction of figure dims) — push content close to edges.
MARGIN_LEFT   = 0.01
MARGIN_RIGHT  = 0.99
MARGIN_TOP    = 0.99
MARGIN_BOTTOM = 0.01

# Inter-row vertical gap (fraction of avg row height). Smaller → header
# sits closer to the waffles, footer hugs the bottom waffle.
HSPACE = 0.05

# Legend label wrap: max chars per line for both labels and section titles.
LEGEND_LABEL_WRAP = 22

# ===== Data =====
data = pd.read_csv(
    '../../Data/Fig3/Talos_solves_VCGS-prospective_261128.csv',
    header=0, index_col=[1, 0], usecols=range(1, 7),
)
data_combined = data.groupby(level=[0, 1]).sum()

CATEGORIES = [
    'Gene-disease Associations',
    'Variant Reclassifications',
    'Missed in Manual Analysis',
    'CNV/SV',
]
ROW_LABELS = [
    'Gene-disease\nAssociations',
    'Variant\nReclassifications',
    'Missed in\nManual Analysis',
    'CNV/SV',
]
COHORTS = [
    # (data column, header text, total diagnoses, cohort N, diagnosis %)
    # Header text is pre-wrapped so each cohort name fits its narrow slot.
    ('All',     'Full\nCohort',          241,  4738, '5.1%'),
    ('NDD',     'NDD\nSub-cohort',        86,  1665, '5.2%'),
    ('Cardiac', 'Cardiac\nSub-cohort',    53,   997, '5.3%'),
    ('Renal',   'Renal\nSub-cohort',      41,   695, '5.9%'),
]

colors_list = [
    '#EC4E20', '#F69A7F', '#FCCCBE',              # Gene-disease (3)
    '#FF9505', '#FFC474',                          # Variant Reclass (2)
    '#016FB9', '#4799D0', '#8EC4E8', '#D4EEFF',   # Missed (4)
    '#57ABA9', '#C2FCFA',                          # CNV/SV (2)
]
COLOR_RANGES = {
    'Gene-disease Associations': (0, 3),
    'Variant Reclassifications': (3, 5),
    'Missed in Manual Analysis': (5, 9),
    'CNV/SV':                    (9, 11),
}

# Cohort relative widths derived from the largest waffle in each cohort,
# so every cohort has the same slot_width / max_cols ratio → identical
# cell size across every waffle in the figure.
def max_cols_for_cohort(cohort_key):
    return max(
        math.ceil(data_combined.loc[cat, cohort_key].sum() / WAFFLE_ROWS)
        for cat in CATEGORIES
    )
COHORT_WIDTHS = {c[0]: max_cols_for_cohort(c[0]) for c in COHORTS}

# ===== Layout (relative units; gridspec normalises) =====
# All width quantities are in "cell-equivalent" units so that
# COHORT_WIDTHS values represent actual cell counts. ROW_LABEL_W and
# LEGEND_W are tuned to give the row-label and legend columns enough
# space at the chosen FIG_WIDTH_MM and FONTSIZE.
ROW_LABEL_W = 5.0    # left text column (label width + right-padding)
LEGEND_W    = 9.0    # right legend column (narrow → labels wrap)
HEADER_H    = 0.55   # ~just enough for 3-line header at HEADER_FONTSIZE
FOOTER_H    = 0.35   # ~just enough for 2-line footer at FONTSIZE
WAFFLE_H    = 1.0

width_ratios  = [ROW_LABEL_W] + [COHORT_WIDTHS[c[0]] for c in COHORTS] + [LEGEND_W]
height_ratios = [HEADER_H] + [WAFFLE_H] * len(CATEGORIES) + [FOOTER_H]

gs = GridSpec(
    nrows=len(CATEGORIES) + 2,
    ncols=len(COHORTS) + 2,
    width_ratios=width_ratios,
    height_ratios=height_ratios,
    left=MARGIN_LEFT, right=MARGIN_RIGHT,
    top=MARGIN_TOP,   bottom=MARGIN_BOTTOM,
    hspace=HSPACE, wspace=0.10,
)

# ===== Waffles =====
# pywaffle's plots dict accepts int / str / tuple keys. Wrapping each
# SubplotSpec in a single-element tuple makes pywaffle call
# `add_subplot(SubplotSpec, aspect='equal')` — letting us drive layout
# from the gridspec above.
# plot_anchor='W' (the pywaffle default) left-aligns each waffle in its
# slot, so the four category waffles within a cohort column share the
# same left edge and can be compared directly.
plots = {}
for col_idx, (cohort_key, *_) in enumerate(COHORTS, start=1):
    for row_idx, cat in enumerate(CATEGORIES, start=1):
        c0, c1 = COLOR_RANGES[cat]
        plots[(gs[row_idx, col_idx],)] = {
            'values': data_combined.loc[cat, cohort_key],
            'colors': colors_list[c0:c1],
        }

fig = plt.figure(
    figsize=(FIG_WIDTH_MM * MM, FIG_HEIGHT_MM * MM),
    FigureClass=Waffle,
    plots=plots,
    rows=WAFFLE_ROWS,
    rounding_rule='ceil',
    tight=False,  # disable pywaffle's tight_layout — incompatible with our gridspec
)

# ===== Text labels =====
def text_axis(spec):
    ax = fig.add_subplot(spec)
    ax.axis('off')
    return ax

# Header row: anchor to BOTTOM of slot (va='bottom', y=0) so the text
# sits flush above the first waffle row, with any unused header height
# becoming top-padding rather than text-to-waffle padding.
text_axis(gs[0, 0]).text(
    0.0, 0.0, 'New Diagnoses\n(Count)',
    ha='left', va='bottom', fontsize=HEADER_FONTSIZE, fontweight='bold',
)

for col_idx, (_, name, count, *_) in enumerate(COHORTS, start=1):
    text_axis(gs[0, col_idx]).text(
        0.5, 0.0, f"{name}\n({count})",
        ha='center', va='bottom', fontsize=HEADER_FONTSIZE,
    )

# Row labels: vertically centred within their waffle row.
for row_idx, label in enumerate(ROW_LABELS, start=1):
    text_axis(gs[row_idx, 0]).text(
        0.0, 0.5, label, ha='left', va='center', fontsize=FONTSIZE,
    )

# Footer row: anchor to TOP of slot (va='top', y=1) so the text sits
# flush below the last waffle row.
text_axis(gs[-1, 0]).text(
    0.0, 1.0, 'Total Cohort Size\nNew Diagnosis %',
    ha='left', va='top', fontsize=FONTSIZE, fontweight='bold',
)

for col_idx, (_, _, _, total, pct) in enumerate(COHORTS, start=1):
    text_axis(gs[-1, col_idx]).text(
        0.5, 1.0, f"N={total}\n{pct}",
        ha='center', va='top', fontsize=FONTSIZE,
    )

# ===== Vertical legend =====
# Render four mini-legends stacked in the right column. We attach them
# to a single axis spanning the legend column, then reposition each
# using its measured rendered height in figure coordinates so they
# stack without overlap regardless of wrap settings.
def wrapped(text):
    return textwrap.fill(text, width=LEGEND_LABEL_WRAP)

ax_leg = fig.add_subplot(gs[1:-1, -1])
ax_leg.axis('off')

# First pass: create each legend at a placeholder position so we can measure.
legends = []
for cat in CATEGORIES:
    c0, c1 = COLOR_RANGES[cat]
    subcats = list(data_combined.loc[cat].index)
    handles = [
        mpatches.Patch(color=colors_list[c0 + j], label=wrapped(s))
        for j, s in enumerate(subcats)
    ]
    leg = ax_leg.legend(
        handles=handles,
        loc='upper left',
        bbox_to_anchor=(0, 1),  # placeholder; repositioned below
        title=wrapped(cat),
        title_fontproperties={'weight': 'bold', 'size': FONTSIZE},
        fontsize=FONTSIZE,
        frameon=False,
        handlelength=1.2,
        handletextpad=0.6,
        borderpad=0,
        labelspacing=0.4,
    )
    # Left-align the title with the swatch column. matplotlib stores the
    # title + entry rows in a VPacker; setting its `align` to "left" makes
    # the title left-justified instead of centered above the entries.
    leg._legend_box.align = "left"
    ax_leg.add_artist(leg)
    legends.append(leg)

# Stack top-down using measured heights in figure-fraction coords.
# (Figure coords avoid clipping at ax_leg's axis bounds.)
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

leg_col_pos = gs[1:-1, -1].get_position(fig)
x_left = leg_col_pos.x0
y_cursor = leg_col_pos.y1
SECTION_GAP_FIG = 0.01  # figure-fraction gap between sections

for leg in legends:
    bbox_px = leg.get_window_extent(renderer)
    height_fig = bbox_px.height / fig.bbox.height
    leg.set_bbox_to_anchor((x_left, y_cursor), transform=fig.transFigure)
    y_cursor -= height_fig + SECTION_GAP_FIG

fig.savefig('../../Figures/Fig3/Fig3_panelA.pdf')
plt.show()


In [ ]:
! open ../../Figures/Fig3/Fig3_panelA.pdf